In [ ]:
# ==================== 1. IMPORT LIBRARIES & UPLOAD DATA ====================
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression
import shap
import warnings
warnings.filterwarnings('ignore')

# Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Read CSV into pandas dataframe
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(f"Successfully loaded file: {file_name}, shape: {df.shape}")

# Remove unnamed index column if exists
if df.columns[0] == 'Unnamed: 0':
    df = df.drop(df.columns[0], axis=1)
    print("Removed unnamed column")


In [ ]:
# ==================== 2. DATA CLEANING & BASIC INFO ====================
print("=== 2. DATA CLEANING & BASIC INFO ===")

# Remove missing values
initial_count = len(df)
df = df.dropna()
print(f"After removing missing values: {len(df)} rows (removed {initial_count - len(df)} rows)")

# Remove duplicate songs based on 'song_id'
df = df.drop_duplicates(subset=['song_id'])
print(f"After removing duplicate song_ids: {len(df)} rows")
print(f"Final data shape: {df.shape}")

# Display basic info
print("\nDataframe info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())


In [ ]:
# ==================== 3. EXPLORATORY DATA ANALYSIS (EDA) ====================
print("=== 3. EXPLORATORY DATA ANALYSIS ===")

# Popularity distribution
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.hist(df['popularity'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel('Popularity Score')
plt.ylabel('Count')
plt.title('Distribution of Song Popularity')

plt.subplot(1,2,2)
sns.boxplot(y=df['popularity'], color='lightcoral')
plt.title('Boxplot of Popularity')
plt.tight_layout()
plt.show()

# Numeric feature correlation
numeric_features = ['duration_ms', 'danceability', 'energy', 'key', 'loudness',
                    'speechiness', 'acousticness', 'instrumentalness', 'liveness',
                    'valence', 'tempo', 'time_signature', 'popularity']
corr_matrix = df[numeric_features].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

print("Correlation with popularity (sorted):")
print(corr_matrix['popularity'].sort_values(ascending=False))

# Categorical analysis: Explicit content & Top genres
fig, axes = plt.subplots(1,2, figsize=(15,6))

# Explicit
explicit_means = df.groupby('explicit')['popularity'].mean()
axes[0].bar(['Non-Explicit','Explicit'], explicit_means.values, color=['lightblue','lightcoral'], alpha=0.8)
axes[0].set_title('Average Popularity by Explicit Content')
axes[0].set_ylabel('Average Popularity')

# Genre (Top 15)
top_genres = df['song_genre'].value_counts().head(15).index
genre_means = df[df['song_genre'].isin(top_genres)].groupby('song_genre')['popularity'].mean().sort_values(ascending=False)
colors = plt.cm.viridis(np.linspace(0,1,len(genre_means)))
axes[1].barh(range(len(genre_means)), genre_means.values, color=colors, alpha=0.8)
axes[1].set_yticks(range(len(genre_means)))
axes[1].set_yticklabels(genre_means.index)
axes[1].set_title('Average Popularity by Genre (Top 15)')
axes[1].set_xlabel('Average Popularity')
plt.tight_layout()
plt.show()


In [ ]:
# ==================== 4. FEATURE ENGINEERING ====================
print("=== 4. FEATURE ENGINEERING ===")

# Create new features
df['duration_min'] = df['duration_ms'] / 60000
df['energy_loudness'] = df['energy'] * df['loudness']
df['dance_valence'] = df['danceability'] * df['valence']

# Numeric features
features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness',
            'liveness','valence','tempo','duration_min','energy_loudness','dance_valence']

# One-hot encode categorical variables
genre_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
genre_encoded = genre_encoder.fit_transform(df[['song_genre']])
genre_columns = [f'genre_{name}' for name in genre_encoder.categories_[0]]

explicit_encoder = OneHotEncoder(sparse_output=False, drop='first')
explicit_encoded = explicit_encoder.fit_transform(df[['explicit']])

# Combine all features
X_numeric = df[features].values
X_combined = np.hstack([X_numeric, genre_encoded, explicit_encoded])
feature_names = features + genre_columns + ['explicit']
y = df['popularity'].values

# Feature selection (SelectKBest)
k_features = min(15, X_combined.shape[1])
selector = SelectKBest(score_func=f_regression, k=k_features)
X_selected = selector.fit_transform(X_combined, y)
selected_mask = selector.get_support()
selected_features = [feature_names[i] for i in range(len(feature_names)) if selected_mask[i]]
print(f"Selected features ({len(selected_features)}): {selected_features}")


In [ ]:
# ==================== 5. MODEL TRAINING & EVALUATION ====================
print("=== 5. MODEL TRAINING & EVALUATION ===")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

# Define models
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=7, random_state=42),
    'Linear Regression': LinearRegression()
}

# Train models & evaluate
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'MSE': mean_squared_error(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }
    print(f"{name}: MSE={results[name]['MSE']:.2f}, MAE={results[name]['MAE']:.2f}, R²={results[name]['R2']:.3f}")

# Select best model
best_model_name = max(results.keys(), key=lambda x: results[x]['R2'])
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name}")


In [ ]:
# ==================== 6. MODEL INTERPRETATION & SHAP ====================
print("=== 6. MODEL INTERPRETATION & SHAP ===")

# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.DataFrame({'Feature': selected_features, 'Importance': best_model.feature_importances_}).sort_values('Importance', ascending=False)
    plt.figure(figsize=(12,8))
    sns.barplot(x='Importance', y='Feature', data=feat_imp.head(15), palette='viridis', alpha=0.8)
    plt.title(f'{best_model_name} - Feature Importance', fontsize=16)
    plt.tight_layout()
    plt.show()
    print("Top 10 features:")
    print(feat_imp.head(10))

# SHAP analysis for tree-based models
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Decision Tree']:
    print("\nPerforming SHAP analysis...")
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)
    plt.figure(figsize=(12,8))
    shap.summary_plot(shap_values, X_test, feature_names=selected_features, plot_type='bar', show=False)
    plt.tight_layout()
    plt.show()


In [ ]:
# ==================== 7. ERROR ANALYSIS & BUSINESS STRATEGIES ====================
print("=== 7. ERROR ANALYSIS ===")

y_pred_best = best_model.predict(X_test)
errors = y_test - y_pred_best

# Residual scatter & histogram
plt.figure(figsize=(14,6))
plt.subplot(1,2,1)
plt.scatter(y_pred_best, errors, alpha=0.6, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Popularity')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Analysis')
plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.hist(errors, bins=30, edgecolor='black', alpha=0.7, color='lightcoral')
plt.xlabel('Prediction Error')
plt.ylabel('Count')
plt.title('Error Distribution')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Large errors
large_error_idx = np.where(np.abs(errors) > 20)[0]
print(f"Number of samples with error > 20: {len(large_error_idx)}")

# Business recommendations (based on top features)
print("\n=== BUSINESS RECOMMENDATIONS ===")
print("""
1. Focus on high impact features for popularity: danceability, energy, loudness
2. Optimize song length (~3-4 minutes) and feature combinations (e.g., dance_valence)
3. Target explicit/non-explicit content based on audience preferences
4. Prioritize promotion for genres with high predicted popularity
5. Use predictive modeling to test feature combinations before release
""")
